# Lecture 6.6 — CAPSTONE: Multi-Agent Research Assistant  ·  STARTER

**Section 06 — Tracing, Observability & Capstone**

> **RECORDING AID — NOT A STUDENT DELIVERABLE.** Cells marked `# LIVE CODE - type during recording` are typed live on camera. All other cells are pre-built and runnable. The student download is `Lecture_6.6_Capstone_COMPLETE.ipynb`.

This is the final lecture of the course. You are going to build a complete
multi-agent research assistant: it validates the topic, plans a set of web
searches, runs those searches in parallel, routes the findings to the right
writer, checks the finished report against a quality bar, and reports what the
whole thing cost. The entire run lands in a single trace you can open in the
OpenAI dashboard.

**Nothing in this notebook is new.** Every class, decorator, parameter and
property used here has already appeared in Sections 1 through 6. The capstone
is not about learning another API surface. It is about wiring the surfaces you
already know into one system that holds together.

## Cell 1: Install the SDK

This cell installs the OpenAI Agents SDK into the current Colab runtime.

Note the `[viz]` extra. That is a small addition to the usual install line. The
final cell of this notebook draws a picture of the agent graph, and the drawing
code depends on `graphviz`, which is not part of the base package. Asking for
`openai-agents[viz]` pulls both in together.

The version is pinned so that every example in this notebook behaves exactly as
recorded. If you would rather track the newest release, drop the `==` and
everything after it.

If the cell finishes without output, the package is installed and ready. If the
package is already present in this runtime, pip will confirm that and move on.
On some Colab runtimes the `graphviz` Python bindings install fine but the
underlying system binary is missing. The commented `apt-get` line is there for
exactly that case.

In [ ]:
# Pinned for reproducibility. To use the latest version,
# run: pip install "openai-agents[viz]"
# Or substitute your preferred version below.
# The [viz] extra installs graphviz, needed by draw_graph below.
!pip install "openai-agents[viz]==0.19.1" -q

# If graphviz is missing when you reach the final cell,
# uncomment the line below:
# !apt-get install -y graphviz -q

## Cell 2: API key setup

The SDK reads your OpenAI credentials from the `OPENAI_API_KEY` environment
variable. This cell pulls the key out of Colab Secrets and writes it there.

**To add the secret in Colab:**

1. Click the key icon (🔑) in the left sidebar.
2. Click **Add new secret**.
3. Set **Name** to `OPENAI_API_KEY`.
4. Paste your key into the **Value** field.
5. Toggle **Notebook access** on for this notebook.

The key never appears in the notebook itself, so you can share the file freely.

**Running locally instead of in Colab?** Skip this cell and set the variable in
your terminal before starting Jupyter: `export OPENAI_API_KEY="sk-..."`.

One thing worth knowing before you run anything: this notebook makes real API
calls, including live web searches. A full run costs a small amount of money.

In [ ]:
import os

from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("API key loaded.")

## Cell 3: Choose the model

Every agent in this notebook reads its model from a single variable. Change
`MODEL_NAME` here and the planner, the searchers, the writers, the triage
agent and both guardrail checkers all switch together. No hunting through cells
for hardcoded strings.

`gpt-5.4-mini` is a good fit for this system. It is fast enough that three
parallel searches feel quick, cheap enough that experimenting is not painful,
and capable enough to plan searches and write a coherent report.

In [ ]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

print(f"Using model: {MODEL_NAME}")

## Cell 4: Imports

Every import in this cell has already been taught. Here is where each one came
from:

| Import | First taught in |
|---|---|
| `Agent`, `Runner`, `ModelSettings` | Sections 1 and 2 |
| `Reasoning` (from `openai.types.shared`) | Lecture 2.2 |
| `WebSearchTool` | Lecture 3.3 |
| `Usage` | Lecture 4.7 |
| `handoff`, `RECOMMENDED_PROMPT_PREFIX` | Lectures 5.2 and 5.5 |
| `input_guardrail`, `output_guardrail`, `GuardrailFunctionOutput` | Lectures 5.8 and 5.9 |
| `InputGuardrailTripwireTriggered`, `OutputGuardrailTripwireTriggered` | Lectures 4.6, 5.8, 5.10 |
| `trace`, `gen_trace_id`, `custom_span` | Lectures 6.2 and 6.3 |
| `RunHooks`, `AgentHookContext` | Lecture 6.4 |
| `draw_graph` | Lecture 6.5 |

Two of these trip people up regularly, so they are worth calling out again.
`Reasoning` comes from `openai.types.shared`, not from `agents`. And
`RECOMMENDED_PROMPT_PREFIX` lives under `agents.extensions.handoff_prompt`,
not at the top level.

Consolidating every import into one cell means you can restart the runtime,
run the first four cells, and jump straight to whichever cell you want to
experiment with.

In [ ]:
import asyncio
from typing import Any

from openai.types.shared import Reasoning
from pydantic import BaseModel, Field

from agents import (
    Agent,
    AgentHookContext,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    ModelSettings,
    OutputGuardrailTripwireTriggered,
    RunContextWrapper,
    RunHooks,
    Runner,
    TResponseInputItem,
    Usage,
    WebSearchTool,
    custom_span,
    gen_trace_id,
    handoff,
    input_guardrail,
    output_guardrail,
    trace,
)
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
from agents.extensions.visualization import draw_graph

print("Imports ready.")

## Cell 5: Architecture overview

Before writing any agents, here is the shape of the system you are building.

**The pipeline, end to end:**

1. A topic arrives. An input guardrail decides whether it is a legitimate
   research question. If it is not, the run stops here.
2. The **planner agent** turns the topic into exactly three web searches,
   returned as a typed Python object.
3. Those three searches run **in parallel**, each in its own agent, each with
   the hosted `WebSearchTool`. The whole search phase is wrapped in a custom
   span so it shows up as one labelled block in the trace.
4. The three summaries plus the original query go to a **triage agent**, which
   hands off to whichever writer suits the topic.
5. The chosen **writer agent** produces a structured report.
6. An output guardrail on the writer checks the report against a quality bar.
7. Everything above happens inside one `trace()` block, so the entire run is a
   single trace with one shareable link.
8. Lifecycle hooks print progress as agents start, finish and hand off.
9. A `Usage` object accumulates token counts across all five model calls.
10. A final cell draws the finished agent graph.

**Where each piece came from:**

| Stage | Component | Taught in |
|---|---|---|
| 0 | Input guardrail, `run_in_parallel=False` | 5.8, 5.10 |
| 1 | Planner agent with `output_type=WebSearchPlan` | 2.5 |
| 2 | Parallel search agents via `asyncio.gather` | 3.3, 5.7 |
| 3 | `custom_span` around the search phase | 6.3 |
| 4 | Triage agent routing via `handoffs` | 5.2, 5.5 |
| 5 | Two writer agents with `output_type=ReportData` | 2.5 |
| 6 | Output guardrail on the writers | 5.9 |
| 7 | Whole flow inside `trace()` with a deep link | 6.2, 6.3 |
| 8 | `RunHooks` for live progress logging | 6.4 |
| 9 | Per-run cost report from `Usage` | 4.7 |
| 10 | `draw_graph` of the finished system | 6.5 |

That table is the defining constraint of a capstone. Nothing new is
introduced. Everything here has been taught.

**A note on scale.** The planner is capped at exactly three searches. Three
searches keep a full run to roughly thirty to sixty seconds and a few cents,
which is the right size for a notebook you will run several times while
experimenting. A production research system would plan considerably more, and
the SDK's own research bot example plans between five and twenty. To raise the
cap, edit the planner's instructions in Cell 8 and change the number. The rest
of the pipeline needs no changes at all, because `asyncio.gather` fans out over
whatever the planner returns. The trade you are making is straightforward:
more searches means broader coverage, more tokens, more money and more waiting.

## Cell 6: The data contracts

Five Pydantic models define every structured handover in this system. Three of
them mirror the SDK's own research bot example. Two are guardrail verdicts.

| Model | Used by | Purpose |
|---|---|---|
| `WebSearchItem` | planner | One search: the term, plus why it is worth running |
| `WebSearchPlan` | planner | The full list of searches |
| `ReportData` | writers | Summary, full markdown report, follow-up questions |
| `TopicCheckOutput` | input guardrail | Is this a real research topic? |
| `ReportQualityOutput` | output guardrail | Does the report meet the bar? |

The `Field(description=...)` calls are not decoration. They become part of the
JSON schema the SDK sends to the model, so they function as instructions
attached directly to each field. Telling the model what `reason` means in the
schema is more reliable than explaining it in a prompt and hoping the model
connects the two.

These types are what make the whole pipeline code-driven rather than
string-driven. Because the planner returns a `WebSearchPlan`, Cell 10 can write
`for item in plan.searches` and get real objects. Because the writers return
`ReportData`, Cell 17 can write `report.markdown_report` with no parsing at
all. Structured outputs from Lecture 2.5 are the load-bearing wall of this
entire notebook.

In [ ]:
class WebSearchItem(BaseModel):
    reason: str = Field(
        description=(
            "Your reasoning for why this search is important "
            "to the query."
        )
    )
    query: str = Field(
        description="The search term to use for the web search."
    )


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(
        description=(
            "A list of web searches to perform to best answer "
            "the query."
        )
    )


class ReportData(BaseModel):
    short_summary: str = Field(
        description="A short 2-3 sentence executive summary."
    )
    markdown_report: str = Field(
        description="The full markdown report."
    )
    follow_up_questions: list[str] = Field(
        description="Suggested topics to research further."
    )


class TopicCheckOutput(BaseModel):
    is_valid_research_topic: bool
    reasoning: str


class ReportQualityOutput(BaseModel):
    meets_standards: bool
    word_count: int
    reasoning: str


print("Data contracts defined.")

## Cell 7: The input guardrail

This is the front door of the system. Before anything expensive happens, a
small, cheap agent reads the incoming topic and decides whether it is a
legitimate research question.

The pattern is exactly the one from Lecture 5.8. A classifier agent with a
narrow job and a boolean output type, wrapped in a function decorated with
`@input_guardrail`. The function returns `GuardrailFunctionOutput`, and
`tripwire_triggered` is the flag that decides whether the run continues.

**The parameter that matters here is `run_in_parallel=False`.**

Lecture 5.10 laid out the trade between the two execution modes:

| Mode | Behaviour | Cost when the tripwire fires |
|---|---|---|
| `run_in_parallel=True` (default) | Guardrail runs alongside the agent | The agent may already have burned tokens and called tools |
| `run_in_parallel=False` | Guardrail completes before the agent starts | Zero. Nothing downstream ever runs |

In a chat assistant, parallel mode is usually right, because latency is what
the user feels and one wasted call is cheap. Here the calculation flips.
Everything downstream of this guardrail is expensive: a planner call, three
parallel web searches, a triage call and a writer call. That is five model
calls, three of them hitting live web search. Paying one extra second of
latency to guarantee that a rejected topic costs you exactly one cheap
classification is an easy trade.

Note also that the guardrail passes `context=ctx.context` through to its inner
run. That keeps the classifier on the same run context as the pipeline it is
protecting.

In [ ]:
# LIVE CODE - type during recording

## Cell 8: The planner agent

Stage one of the pipeline. The planner reads a research topic and returns a
list of web searches, each with a stated reason.

Two things on this agent are doing real work.

**`output_type=WebSearchPlan`.** Without it, the planner would hand back a
paragraph of prose containing some search suggestions, and you would be writing
a parser. With it, the SDK returns an actual `WebSearchPlan` object, and the
next stage can iterate over `plan.searches` directly. This is what turns a
conversation into a program.

**`input_guardrails=[validate_topic]`.** The guardrail attaches here rather
than anywhere else because the planner is the entry point of the run. From
Lecture 5.8: input guardrails run only for the first agent in a run. Attaching
this guardrail to the search agent or the triage agent would look reasonable in
the code and would silently never fire.

The instruction to output exactly three searches is the cost cap from Cell 5.
Change the number there and the whole pipeline scales with it.

In [ ]:
# LIVE CODE - type during recording

## Cell 9: The search agent

One agent, one search term, one summary. Three copies of this agent will run
at the same time in the next cell.

`WebSearchTool()` is the hosted tool from Lecture 3.3. Hosted means it runs on
OpenAI's servers rather than in this runtime, so there is no second API key to
configure, no scraping library to install and nothing to maintain. It takes no
required parameters. It only works with OpenAI models, which is fine here
because `MODEL_NAME` is one.

Read the instructions carefully, because they are written for an unusual
reader. Nobody is going to see these summaries. They exist only to be pasted
into a prompt for the writer agent further down the pipeline. That is why they
ask for terseness over polish and essence over completeness. When an agent's
output feeds another agent rather than a human, write the instructions for the
machine that will actually read it.

In [ ]:
# LIVE CODE - type during recording

## Cell 10: Running the searches in parallel

This function takes a plan and returns the summaries. Two ideas from earlier
sections are stacked here.

**`asyncio.gather` (Lecture 5.7).** Building a list of coroutines and awaiting
them together means three searches take as long as the slowest one, not the sum
of all three. On this pipeline that is the difference between roughly twelve
seconds and roughly thirty-six. `gather` also returns results in the order the
coroutines were passed, not the order they finished, so summary one always
corresponds to search one.

**`custom_span` (Lecture 6.3).** Wrapping the whole fan-out in a named span
means the trace shows one labelled block called `parallel_web_searches` instead
of three loose agent spans floating at the top level. You can then see at a
glance how much of the total run time went to searching versus writing. The
`data` dictionary attaches arbitrary metadata to the span, which is handy when
you want to know how many searches a given run actually planned.

There is a catch with `custom_span` worth stating plainly. Outside an active
trace it returns a no-op span that silently records nothing. No error, no
warning, just no data. It works here only because this function is always
called from inside the `trace()` block in Cell 15. If you call `run_searches`
on its own, the span vanishes.

The `usage` parameter is a plain `Usage` object passed in by the caller. Each
search result contributes its tokens to it, so the cost report at the end
covers the searches too rather than quietly leaving them out.

In [ ]:
# LIVE CODE - type during recording

## Cell 11: The writer agents

Two writers, same output type, different editorial instincts. The triage agent
in Cell 13 will pick between them.

Three details are worth pausing on.

**`handoff_description`.** This is the one-line summary the triage agent reads
when deciding where to route. The triage agent never sees these full
instruction strings. It sees the agent name and this description, and that is
what it routes on. A vague `handoff_description` produces vague routing.

**`RECOMMENDED_PROMPT_PREFIX`.** Both writers open with it. From Lecture 5.5,
this prefix tells the model it is part of a multi-agent system and that
transfers between agents happen quietly in the background. Without it, a model
that has just been handed a conversation tends to greet the user or comment on
the transfer.

**The output guardrail is not here.** It is attached in Cell 12, after the
guardrail function itself exists. Python needs the function defined before it
can be referenced, and the guardrail needs `ReportData` and a checker agent.
Attaching it in the next cell keeps the ordering honest. The writers are not
unguarded, they are simply guarded one cell later.

In [ ]:
technical_writer = Agent(
    name="Technical Writer",
    handoff_description=(
        "Writes reports on engineering, science and technology topics."
    ),
    instructions=(
        f"{RECOMMENDED_PROMPT_PREFIX}\n"
        "You are a technical writer. You will be given an original "
        "research query and a set of web search summaries. Write a "
        "thorough report aimed at a technically literate reader. "
        "Emphasise mechanisms, specifications, quantitative detail "
        "and engineering trade-offs. Explain why things work the "
        "way they do, not just that they do. Produce three things: "
        "a short executive summary of two to three sentences, a "
        "full report in markdown with clear headings, and a list "
        "of follow-up questions worth researching next."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    output_type=ReportData,
)

business_writer = Agent(
    name="Business Writer",
    handoff_description=(
        "Writes reports on market, industry and commercial topics."
    ),
    instructions=(
        f"{RECOMMENDED_PROMPT_PREFIX}\n"
        "You are a business writer. You will be given an original "
        "research query and a set of web search summaries. Write a "
        "thorough report aimed at a commercially minded reader. "
        "Emphasise market dynamics, the competitive landscape, "
        "adoption drivers and strategic implications. Explain what "
        "the findings mean for someone making a decision. Produce "
        "three things: a short executive summary of two to three "
        "sentences, a full report in markdown with clear headings, "
        "and a list of follow-up questions worth researching next."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    output_type=ReportData,
)


print("Writer agents ready.")

## Cell 12: The output guardrail

The mirror image of Cell 7. Same shape, one important difference in the
signature and one important difference in placement.

**The signature.** An output guardrail's third parameter is `output`, not
`input`. It receives what the agent produced, after the agent has finished.
That is also why `OutputGuardrail` has no `run_in_parallel` parameter at all.
There is nothing to run in parallel with, because the work is already done by
the time the guardrail sees anything.

**The placement.** These guardrails attach to the two writers, not to the
triage agent. From Lecture 5.9: output guardrails run only on the **last**
agent in a run. In this pipeline the triage agent always hands off, so a writer
is always last, so the writers are where the guardrail has to live.

Worth being honest about the edge case. If the triage agent ignored its
instructions and answered directly instead of handing off, triage would be the
last agent, triage has no output guardrails, and nothing would be checked. The
run would succeed and the quality gate would silently not exist. That is why
the triage instructions in Cell 13 say "Always hand off" in as many words.

The `getattr` call is defensive. `output` arrives as a `ReportData` object in
normal operation, but falling back to `str(output)` means the guardrail cannot
crash the run over an unexpected shape.

Attaching guardrails after construction works because `Agent.output_guardrails`
is an ordinary mutable list attribute.

In [ ]:
# LIVE CODE - type during recording

## Cell 13: The triage agent

This is the only place in the whole system where the model, not your code,
decides what happens next. Cells 1 through 12 are deterministic: you call the
planner, then you call the searches, then you call triage. Here the model reads
the material and picks a writer.

`handoff()` with `tool_name_override` gives each route a name that reads well
in a trace. The default tool name would be `transfer_to_technical_writer`,
derived from the agent name. `route_to_technical_writer` says what is actually
happening in this pipeline. Cosmetic in isolation, genuinely useful when you
are scanning a trace with five spans in it.

The instructions do two jobs. `RECOMMENDED_PROMPT_PREFIX` teaches the model
what a handoff is. "Do not write the report yourself. Always hand off." is not
stylistic advice. It is the instruction that keeps the output guardrail from
Cell 12 in the execution path, for the reason spelled out there.

This is the LLM-driven orchestration pattern from Lecture 5.1. You built the
routes. The model chooses among them.

In [ ]:
# LIVE CODE - type during recording

## Cell 14: Progress logging with RunHooks

A run of this pipeline takes thirty to sixty seconds. Without any output, that
is a long time staring at a spinner with no idea whether anything is working.

The SDK's own research bot example solves this with a `rich`-based progress
display. This notebook uses `RunHooks` from Lecture 6.4 instead. That avoids an
extra dependency and puts a course concept to real use.

`RunHooks` observes an entire `Runner.run()` call, including handoffs, which is
exactly the scope needed here. `AgentHooks` would attach to one agent at a
time. The method names differ between the two and it is a common mix-up:
`RunHooks` uses `on_agent_start` and `on_agent_end`, while `AgentHooks` uses
`on_start` and `on_end`.

The context type differs by event too. Agent start and end hooks receive an
`AgentHookContext`, which carries the shared run usage state, so
`context.usage.total_tokens` gives a running total as the pipeline progresses.
The handoff hook receives a plain `RunContextWrapper`.

**No tool hook here, and that is deliberate.** `on_tool_start` and
`on_tool_end` fire for local tools. `WebSearchTool` is a hosted tool that runs
on OpenAI's servers, so it does not go through that path and those hooks would
never fire for it. Adding them would produce a hook that looks correct, runs
silently and teaches you nothing.

In [ ]:
class ResearchProgressHooks(RunHooks):
    def __init__(self) -> None:
        self.steps: list[str] = []

    async def on_agent_start(
        self,
        context: AgentHookContext,
        agent: Agent,
    ) -> None:
        msg = f"  -> {agent.name} started"
        self.steps.append(msg)
        print(msg, flush=True)

    async def on_agent_end(
        self,
        context: AgentHookContext,
        agent: Agent,
        output: Any,
    ) -> None:
        msg = (
            f"  <- {agent.name} finished "
            f"({context.usage.total_tokens} tokens so far)"
        )
        self.steps.append(msg)
        print(msg, flush=True)

    async def on_handoff(
        self,
        context: RunContextWrapper,
        from_agent: Agent,
        to_agent: Agent,
    ) -> None:
        msg = f"  >> HANDOFF {from_agent.name} to {to_agent.name}"
        self.steps.append(msg)
        print(msg, flush=True)


print("Progress hooks ready.")

## Cell 15: The orchestrator

This is where the twelve pieces above become one system. It is the longest cell
in the notebook and every line of it is something you have already met.

**Reading it top to bottom:**

`gen_trace_id()` produces a properly formatted trace ID up front, before
anything runs. That matters because it lets you print the dashboard link
immediately. You can open the trace in another tab and watch spans appear while
the run is still going, rather than waiting for it to finish to find out where
it went wrong.

`with trace("Research assistant", trace_id=trace_id):` is the piece that makes
this one trace instead of five. When `Runner.run()` executes inside an active
trace, it does not create a trace of its own, it joins the one already running.
Three `Runner.run()` calls, one trace, one link, the whole story in a single
view.

`LAST_RUN_USAGE` is a module-level `Usage` object, reset at the top of each
call. Every stage adds its own usage into it. `Usage.add()` aggregates
requests, input tokens, output tokens and totals, which is what makes a
five-call pipeline reportable as one number in Cell 17.

`final_output_as(WebSearchPlan)` casts the planner's result to the type you
expect. By default this is a typechecker convenience rather than a runtime
check. Pass `raise_if_incorrect_type=True` if you want it to actually raise.

`report_result.last_agent.name` tells you which writer produced the report.
Your code never decided that. The model read the summaries and routed.

The two `except` blocks catch the tripwires. `e.guardrail_result.output.output_info`
gives you back the exact object your guardrail function returned, so
`info.reasoning` is the classifier's own explanation of why it blocked the run.
Catching both means a blocked run returns `None` cleanly instead of dumping a
traceback.

In [ ]:
# LIVE CODE - type during recording

## Cell 16: Run it

The moment of truth. This makes real API calls including three live web
searches, and takes roughly thirty to sixty seconds.

**What to watch for, in order:**

1. The trace link prints first, before any work happens. Open it in another tab.
2. The three planned search terms appear. Notice they are genuinely different
   angles on the question, not three rewordings of it.
3. The hooks start firing. Agents start and finish, and the running token count
   climbs with each one.
4. A `>> HANDOFF` line appears. That is the triage agent choosing a writer.
5. The final line names the writer that produced the report.

Once it finishes, open that trace link. You will see the whole run as one tree,
with the `parallel_web_searches` span sitting there as a single labelled block
containing three concurrent children.

**Want to see the guardrail fire?** Re-run this cell with something that is not
a research topic, such as `"write me a poem about my cat"`, and watch the run
stop at the input guardrail before a single search happens.

In [ ]:
# LIVE CODE - type during recording

## Cell 17: The report, and what it cost

Two payoffs in one short cell.

**The report.** `report.markdown_report` is just an attribute access. No
parsing, no regular expressions, no hoping the model wrapped its output in the
fences you asked for. The writer declared `output_type=ReportData`, so what
came back is a `ReportData` object. That is structured outputs from Lecture 2.5
paying off at the very end of the pipeline.

**The cost.** `LAST_RUN_USAGE` accumulated usage from every stage: one planner
call, three search calls and however many calls triage plus the chosen writer
took. `requests` is the count of model calls, which is a useful sanity check
against what you expected the pipeline to do. If that number surprises you, the
trace will tell you why.

Token counts are the raw material for a cost estimate. Multiply by the
per-token price of your model and you have the real figure for one research
run, which is exactly what you need before letting a system like this loose on
real traffic.

In [ ]:
# LIVE CODE - type during recording

## Cell 18: Draw the system

One line, and you get a picture of what you built.

`draw_graph` has to be the last expression in the cell for the image to render
inline. Assign it to a variable and nothing appears.

One honest caveat, and it is the gotcha from Lecture 6.5. The routes here were
built with `handoff()` objects rather than by passing the writer agents
directly. When the visualiser walks the graph it draws a node and an edge for
each `handoff()` target, but it does not recurse into that agent. So you will
see triage routing to both writers, and you will not see anything inside the
writers. Pass bare agents in `handoffs=[...]` and the visualiser recurses; pass
`handoff()` objects and it stops at the boundary. Neither is wrong, but knowing
which one you are looking at saves you from concluding your graph is broken.

In [ ]:
# LIVE CODE - type during recording

## Cell 19: What you built

Every component in this notebook, and the lecture it came from:

| Component | Lecture |
|---|---|
| `Agent`, `Runner`, `await Runner.run()` | 1.6, 2.1 |
| `ModelSettings`, `Reasoning(effort=...)`, `verbosity` | 2.2 |
| `output_type` and Pydantic structured outputs | 2.5 |
| `WebSearchTool` as a hosted tool | 3.3 |
| `final_output_as()`, `last_agent` | 4.1 |
| Guardrail tripwire exceptions | 4.6 |
| `Usage` and token accounting | 4.7 |
| LLM-driven orchestration | 5.1 |
| `handoff()` and `tool_name_override` | 5.2 |
| `RECOMMENDED_PROMPT_PREFIX` | 5.5 |
| Parallelisation with `asyncio.gather` | 5.7 |
| `@input_guardrail` | 5.8 |
| `@output_guardrail` and last-agent semantics | 5.9 |
| `run_in_parallel=False` and the cost trade | 5.10 |
| Traces and the dashboard | 6.1, 6.2 |
| `trace()` and `custom_span()` | 6.3 |
| `RunHooks` lifecycle logging | 6.4 |
| `draw_graph()` | 6.5 |

Eighteen rows. One system. Nothing new.

---

### Where to take it next

Each of these is buildable with what you already know:

- **Add a `FileSearchTool`** over a vector store of your own documents, so the
  writer can draw on internal material alongside public web results. Same
  hosted-tool pattern as `WebSearchTool`, from Lecture 3.3.
- **Feed in a PDF as baseline context**, so the research starts from a document
  you supply rather than from a bare query.
- **Add an evaluator loop** using the pattern from Lecture 5.6. Let a critic
  agent read the draft and send it back for revision until it passes.
- **Add a `CodeInterpreterTool`** from Lecture 3.3 so the writer can compute and
  chart figures rather than describing them.
- **Raise the search cap** from three to something larger and watch the trace
  to see where the time actually goes.

You have the whole toolkit now. Go build something.